# Data Processing and Visualization
-----
**Assignment Topic: Data Analysis on Car Advertisements**


Student1 Name: Jarrah F.

SNumber: S5212509

Student2 Name: Aditya S.

SNumber: ...

Student3 Name: Grace T.

SNumber: s5407687





I've placed 1D earlier than its actual order since it makes the most sense to first normalise all strings to lower case prior to doing anything else

# Data Import, Tool Import and Variables

In [1]:
# Tool Import
import pandas, seaborn, sqlite3
import matplotlib.pyplot as plt

# Loads the initial Data sets into variables
dfCarData = pandas.read_csv("car_dataset.csv")
dfSellerData = pandas.read_csv("seller_dataset.csv")
dfCarData.rename(columns={"price (AUD)": "price"}, inplace=True)

# Functions

In [20]:
# First 3 functions are not used anymore but were previously part of my IDE script for testing, kept them in for reference but they are unused
# Prints first 5 rows of each dataset to check they loaded properly
def PrintFirst5():
    print(f"\nFirst 5 Rows of Car Dataset \n{dfCarData.head()}")
    print(f"\nFirst 5 Rows of Seller Dataset \n{dfSellerData.head()}\n\n")

# Function prints the duplicate count for both data sets
def CheckForDuplicates():
    print(f"\nFound {dfCarData.duplicated().sum()} duplicates in the car data")
    print(f"\nFound {dfSellerData.duplicated().sum()} duplicates in the seller data")

# Function prints the null count and columns with null counts + their count for both datasets
def CheckForNulls():
    carDataNullCount = dfCarData.isnull().sum().sum()
    sellerDataNullCount = dfSellerData.isnull().sum().sum()
    print(f"\nFound {carDataNullCount} null values in the car data \n")
    if carDataNullCount > 0:
        nulls = dfCarData.isnull().sum()
        print(nulls[nulls > 0].to_string())
    print(f"\nFound {sellerDataNullCount} null values in the seller data\n")
    if sellerDataNullCount > 0:
        nulls = dfSellerData.isnull().sum()
        print(nulls[nulls > 0].to_string())

# This function does the conversion by iterating through the column, extracting any numbers and then setting 
# The value to the extracted numbers, due to the column having a Dtype of str initially i cast it as object so that it doesnt
# Care what the actual type is then finally cast the column as an int64 - 64 bit integer
# Could have written it as a real function that takes in a dataframe and column name an returns the int but didnt find it
# Necessary for the assignment but would be a better approach if the dataset was variable or wanted a general function for it
def CarNumericalConversion():
    dfCarData["mileage"] = dfCarData["mileage"].astype("object")
    for i, value in dfCarData["mileage"].items():
        convertedNum = "".join(n for n in str(value) if n.isdigit())
        dfCarData.at[i, "mileage"] = int(convertedNum)
    dfCarData["mileage"] = dfCarData["mileage"].astype("Int64")
    dfCarData["num_of_doors"] = dfCarData["num_of_doors"].astype("object")
    for i, value in dfCarData["num_of_doors"].items():
        convertedNum = "".join(n for n in str(value) if n.isdigit())
        dfCarData.at[i, "num_of_doors"] = int(convertedNum)
    dfCarData["num_of_doors"] = dfCarData["num_of_doors"].astype("Int64")
    dfCarData["seating_capacity"] = dfCarData["seating_capacity"].astype("object")
    for i, value in dfCarData["seating_capacity"].items():
        convertedNum = "".join(n for n in str(value) if n.isdigit())
        dfCarData.at[i, "seating_capacity"] = int(convertedNum)
    dfCarData["seating_capacity"] = dfCarData["seating_capacity"].astype("Int64")
    dfCarData["fuel_consumption"] = dfCarData["fuel_consumption"].astype("object")
    for i, value in dfCarData["fuel_consumption"].items():
        convertedNum = "".join(n for n in str(value) if n.isdigit())
        dfCarData.at[i, "fuel_consumption"] = int(convertedNum)
    dfCarData["fuel_consumption"] = dfCarData["fuel_consumption"].astype("Int64")
    print(dfCarData.head())
    dfCarData.info()

def EngineFeatureSplit():
    dfCarData.insert(loc=10, column="type_of_engine", value=None)
    dfCarData.insert(loc=11, column="engine_capacity", value=None)

    for i in dfCarData.index:
        engineValue = dfCarData.at[i, "engine"]
        engineCapacity = "".join(n for n in str(engineValue) if n.isdigit() or n == ".")
        engineCapacity = float(engineCapacity) if engineCapacity else None
        dfCarData.at[i, "engine_capacity"] = engineCapacity
        if str(engineValue).lower().__contains__("petrol"):
            dfCarData.at[i, "type_of_engine"] = "Petrol"
        elif str(engineValue).lower().__contains__("diesel"):
            dfCarData.at[i, "type_of_engine"] = "Diesel"
        elif str(engineValue).lower().__contains__("hybrid"):
            dfCarData.at[i, "type_of_engine"] = "Hybrid"
        elif str(engineValue).lower().__contains__("electric"):
            dfCarData.at[i, "type_of_engine"] = "Electric"
        else:
            dfCarData.at[i, "type_of_engine"] = "Unknown"
    dfCarData["engine_capacity"] = dfCarData["engine_capacity"].astype("float")
    dfCarData["type_of_engine"] = dfCarData["type_of_engine"].astype("str")
    print(dfCarData.head(10))
    dfCarData.info()


def HotEncode():
    # We do a check first to see if the column exists if not create it --> was mainly used during testing to repeatedly run this function and check outputs for ordering otherwise just creating without a check works
    if "origin_code" not in dfCarData.columns:         dfCarData.insert(loc=2, column="origin_code", value=None)
    if "condition_code" not in dfCarData.columns:      dfCarData.insert(loc=4, column="condition_code", value=None)
    if "car_model_code" not in dfCarData.columns:      dfCarData.insert(loc=6, column="car_model_code", value=None)
    if "exterior_color_code" not in dfCarData.columns: dfCarData.insert(loc=9, column="exterior_color_code", value=None)
    if "interior_color_code" not in dfCarData.columns: dfCarData.insert(loc=11, column="interior_color_code", value=None)
    if "type_of_engine_code" not in dfCarData.columns: dfCarData.insert(loc=16, column="type_of_engine_code", value=None)
    if "fuel_system_code" not in dfCarData.columns:    dfCarData.insert(loc=19, column="fuel_system_code", value=None)
    if "transmission_code" not in dfCarData.columns:   dfCarData.insert(loc=21, column="transmission_code", value=None)
    if "drive_type_code" not in dfCarData.columns:     dfCarData.insert(loc=23, column="drive_type_code", value=None)
    if "brand_code" not in dfCarData.columns:          dfCarData.insert(loc=26, column="brand_code", value=None)
    if "grade_code" not in dfCarData.columns:          dfCarData.insert(loc=28, column="grade_code", value=None)

    # We then create the encoded value for each column using pandas functions
    dfCarData["origin_code"] = dfCarData["origin"].astype("category").cat.codes
    dfCarData["condition_code"] = dfCarData["condition"].astype("category").cat.codes
    dfCarData["car_model_code"] = dfCarData["car_model"].astype("category").cat.codes
    dfCarData["exterior_color_code"] = dfCarData["exterior_color"].astype("category").cat.codes
    dfCarData["interior_color_code"] = dfCarData["interior_color"].astype("category").cat.codes
    dfCarData["type_of_engine_code"] = dfCarData["type_of_engine"].astype("category").cat.codes
    dfCarData["fuel_system_code"] = dfCarData["fuel_system"].astype("category").cat.codes
    dfCarData["transmission_code"] = dfCarData["transmission"].astype("category").cat.codes
    dfCarData["drive_type_code"] = dfCarData["drive_type"].astype("category").cat.codes
    dfCarData["brand_code"] = dfCarData["brand"].astype("category").cat.codes
    dfCarData["grade_code"] = dfCarData["grade"].astype("category").cat.codes

    # we then create dicts to map these values so we can convert between the code and label
    origin_decoder = dfCarData[["origin_code", "origin"]].drop_duplicates().set_index("origin_code")["origin"].to_dict()
    condition_decoder = dfCarData[["condition_code", "condition"]].drop_duplicates().set_index("condition_code")["condition"].to_dict()
    model_decoder = dfCarData[["car_model_code", "car_model"]].drop_duplicates().set_index("car_model_code")["car_model"].to_dict()
    exterior_color_decoder = dfCarData[["exterior_color_code", "exterior_color"]].drop_duplicates().set_index("exterior_color_code")["exterior_color"].to_dict()
    interior_color_decoder = dfCarData[["interior_color_code", "interior_color"]].drop_duplicates().set_index("interior_color_code")["interior_color"].to_dict()
    engine_type_decoder = dfCarData[["type_of_engine_code", "type_of_engine"]].drop_duplicates().set_index("type_of_engine_code")["type_of_engine"].to_dict()
    fuel_system_decoder = dfCarData[["fuel_system_code", "fuel_system"]].drop_duplicates().set_index("fuel_system_code")["fuel_system"].to_dict()
    transmission_decoder = dfCarData[["transmission_code", "transmission"]].drop_duplicates().set_index("transmission_code")["transmission"].to_dict()
    drive_decoder = dfCarData[["drive_type_code", "drive_type"]].drop_duplicates().set_index("drive_type_code")["drive_type"].to_dict()
    brand_decoder = dfCarData[["brand_code", "brand"]].drop_duplicates().set_index("brand_code")["brand"].to_dict()
    grade_decoder = dfCarData[["grade_code", "grade"]].drop_duplicates().set_index("grade_code")["grade"].to_dict()

    # Print all the decoders for reference
    print(f"Origin Decoder \n {dict(sorted(origin_decoder.items()))}")
    print(f"Condition Decoder \n {dict(sorted(condition_decoder.items()))}")
    print(f"Model Decoder \n {dict(sorted(model_decoder.items()))}")
    print(f"Exteriror Color Decoder \n {dict(sorted(exterior_color_decoder.items()))}")
    print(f"Interiror Color Decoder \n {dict(sorted(interior_color_decoder.items()))}")
    print(f"Engine Type Decoder \n {dict(sorted(engine_type_decoder.items()))}")
    print(f"Fuel System Decoder \n {dict(sorted(fuel_system_decoder.items()))}")
    print(f"Transmission Decoder \n {dict(sorted(transmission_decoder.items()))}")
    print(f"Drive Decoder \n {dict(sorted(drive_decoder.items()))}")
    print(f"Brand Decoder \n {dict(sorted(brand_decoder.items()))}")
    print(f"Grade Decoder \n {dict(sorted(grade_decoder.items()))}")

    # Print a sample of the data plus the dataframe info
    print(dfCarData.head(10))
    dfCarData.info()



def StandardiseText():
    for col in dfCarData.select_dtypes(include=["string"]).columns:
        dfCarData[col] = dfCarData[col].str.lower()
    print(dfCarData.head(10))
    dfCarData.info()


def NormaliseNumericalValues():
    columns = ["engine_capacity", "num_of_doors", "seating_capacity", "fuel_consumption", "price"]
    
    for col in columns:
        min_val = dfCarData[str(col)].min()
        max_val = dfCarData[str(col)].max()
        columName = col + "_minmax_normalised"
        dfCarData[str(columName)] = (dfCarData[col] - min_val) / (max_val - min_val)

    print(dfCarData.head(10))
    dfCarData.info()


def StoreToSQLDatabase():
    # Establishes a connection to the sqlite databases
    # Wanted 2 copies incase other non requested data could be useful for display
    # so we have 2 tables 
    # Car Data - the full set
    # Car Data Refined - the requested data
    conn = sqlite3.connect("database.db")
    cur = conn.cursor()

    
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cur.fetchall()]
    for t in tables:
        cur.execute(f'DELETE FROM "{t}"')
    conn.commit()

    dfCarData.to_sql("Car Data", conn, if_exists="append", index=False)
    dfSellerData.to_sql("Seller Data", conn, if_exists="append", index=False)
    table = "Car Data Refined"
    cur.execute(f'PRAGMA table_info("{table}")')
    db_cols = [row[1] for row in cur.fetchall()]
    matched_cols = [c for c in dfCarData.columns if c in db_cols]
    dfCarData[matched_cols].to_sql(table, conn, if_exists="append", index=False)
    conn.close()

    
    

# Data Formatting

## 1D

In [3]:
StandardiseText()

    ad_id             origin condition    car_model mileage exterior_color  \
0   17042  domestic assembly   new car        truck    0 km          white   
1   53794           imported   new car          suv    0 km          black   
2   73954  domestic assembly   new car    crossover    0 km         silver   
3   74150           imported   new car          suv    0 km          white   
4   87573  domestic assembly   new car    crossover    0 km         silver   
5   97011  domestic assembly   new car  van/minivan    0 km          white   
6  101726  domestic assembly   new car          suv    0 km          white   
7  135739           imported   new car          suv    0 km         copper   
8  142495  domestic assembly   new car          4x4    0 km           grey   
9  143308  domestic assembly   new car          4x4    0 km          black   

  interior_color num_of_doors seating_capacity         engine  \
0           gray       2-door           2-seat  petrol\t1.0 l   
1          

## 1A

In [4]:
CarNumericalConversion()

   ad_id             origin condition  car_model  mileage exterior_color  \
0  17042  domestic assembly   new car      truck        0          white   
1  53794           imported   new car        suv        0          black   
2  73954  domestic assembly   new car  crossover        0         silver   
3  74150           imported   new car        suv        0          white   
4  87573  domestic assembly   new car  crossover        0         silver   

  interior_color  num_of_doors  seating_capacity         engine fuel_system  \
0           gray             2                 2  petrol\t1.0 l         NaN   
1          black             5                 7  petrol\t3.4 l         NaN   
2          brown             5                 8  petrol\t2.0 l         NaN   
3          black             5                 5  petrol\t1.8 l         NaN   
4           gray             5                 8  petrol\t2.0 l         NaN   

  transmission                 drive_type  fuel_consumption   brand 

## 1B

In [5]:
EngineFeatureSplit()

    ad_id             origin condition    car_model  mileage exterior_color  \
0   17042  domestic assembly   new car        truck        0          white   
1   53794           imported   new car          suv        0          black   
2   73954  domestic assembly   new car    crossover        0         silver   
3   74150           imported   new car          suv        0          white   
4   87573  domestic assembly   new car    crossover        0         silver   
5   97011  domestic assembly   new car  van/minivan        0          white   
6  101726  domestic assembly   new car          suv        0          white   
7  135739           imported   new car          suv        0         copper   
8  142495  domestic assembly   new car          4x4        0           grey   
9  143308  domestic assembly   new car          4x4        0          black   

  interior_color  num_of_doors  seating_capacity         engine  ...  \
0           gray             2                 2  petrol\t

## 1C

In [6]:
HotEncode()

Origin Decoder 
 {0: 'domestic assembly', 1: 'imported'}
Condition Decoder 
 {0: 'new car', 1: 'used car'}
Model Decoder 
 {0: '4x4', 1: 'convertible/cabriolet', 2: 'coupe', 3: 'crossover', 4: 'hatchback', 5: 'sedan', 6: 'suv', 7: 'truck', 8: 'van/minivan', 9: 'wagon'}
Exteriror Color Decoder 
 {0: '-', 1: 'black', 2: 'brown', 3: 'colorful', 4: 'copper', 5: 'cream', 6: 'different color', 7: 'green', 8: 'grey', 9: 'orange', 10: 'pink', 11: 'red', 12: 'sand', 13: 'silver', 14: 'take note', 15: 'violet', 16: 'white', 17: 'yellow'}
Interiror Color Decoder 
 {0: '-', 1: 'black', 2: 'brown', 3: 'colorful', 4: 'copper', 5: 'cream', 6: 'different color', 7: 'gray', 8: 'green', 9: 'grey', 10: 'orange', 11: 'pink', 12: 'red', 13: 'sand', 14: 'silver', 15: 'violet', 16: 'white', 17: 'yellow'}
Engine Type Decoder 
 {0: 'Diesel', 1: 'Electric', 2: 'Hybrid', 3: 'Petrol', 4: 'Unknown'}
Fuel System Decoder 
 {-1: nan, 0: ' 1.5l', 1: ' 2.0 turbo', 2: ' 2.0l i4 tdci bi-turbo diesel engine', 3: ' 2.4le',

## 1E

In [7]:
NormaliseNumericalValues()

    ad_id             origin  origin_code condition  condition_code  \
0   17042  domestic assembly            0   new car               0   
1   53794           imported            1   new car               0   
2   73954  domestic assembly            0   new car               0   
3   74150           imported            1   new car               0   
4   87573  domestic assembly            0   new car               0   
5   97011  domestic assembly            0   new car               0   
6  101726  domestic assembly            0   new car               0   
7  135739           imported            1   new car               0   
8  142495  domestic assembly            0   new car               0   
9  143308  domestic assembly            0   new car               0   

     car_model  car_model_code  mileage exterior_color  exterior_color_code  \
0        truck               7        0          white                   16   
1          suv               6        0          black      

# Data Exploration and Cleaning

## 2A

,ad_id,origin_code,condition_code,car_model_code,mileage,exterior_color_code,interior_color_code,num_of_doors,seating_capacity,type_of_engine_code,...,fuel_consumption,brand_code,grade_code,year_of_manufacture,price,engine_capacity_minmax_normalised,num_of_doors_minmax_normalised,seating_capacity_minmax_normalised,fuel_consumption_minmax_normalised,price_minmax_normalised
count,3.065200e+04,30652.000000,30652.000000,30652.000000,30652.0,30652.000000,30652.000000,30652.0,30652.0,30652.000000,...,30652.0,30652.000000,30652.000000,30620.000000,3.065200e+04,29285.000000,30652.0,30652.0,30652.0,30652.000000
mean,4.784121e+06,0.417265,0.801971,4.794532,412315.032755,10.021956,3.511092,4.502969,5.539997,2.449106,...,13400.339293,42.403758,224.220573,2017.320346,4.284316e+04,0.154511,0.083388,0.117872,0.000067,0.019597
std,3.103376e+05,0.493115,0.398521,1.784387,34899663.670771,5.918471,3.468700,0.952747,1.582484,1.147354,...,1279212.604754,18.507637,135.306755,5.331772,7.858559e+04,0.064822,0.017643,0.03367,0.006396,0.035946
min,1.704200e+04,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,...,100.0,0.000000,0.000000,1990.000000,4.050000e-05,0.000000,0.0,0.0,0.0,0.000000
25%,4.807684e+06,0.000000,1.000000,4.000000,0.0,4.000000,1.000000,4.0,5.0,3.000000,...,100.0,30.000000,106.000000,2015.000000,1.639680e+04,0.111111,0.074074,0.106383,0.0,0.007500
50%,4.864924e+06,0.000000,1.000000,5.000000,20000.0,11.000000,2.000000,5.0,5.0,3.000000,...,100.0,44.000000,218.000000,2019.000000,2.493933e+04,0.150794,0.092593,0.106383,0.0,0.011407
75%,4.903282e+06,1.000000,1.000000,6.000000,60000.0,16.000000,5.000000,5.0,7.0,3.000000,...,6100.0,52.000000,340.250000,2022.000000,4.044543e+04,0.182540,0.092593,0.148936,0.00003,0.018500
max,4.930147e+06,1.000000,1.000000,9.000000,4294967295.0,17.000000,17.000000,54.0,47.0,4.000000,...,200000100.0,75.000000,478.000000,2023.000000,2.186240e+06,1.000000,1.0,1.0,1.0,1.000000


## 2B

# Data Storage

## 3: Store to the Database

In [21]:
StoreToSQLDatabase()

## 3: Queries

In [24]:
conn = sqlite3.connect("database.db")
dfCarSummary = pandas.read_sql("SELECT * FROM car_summary", conn)
print(dfCarSummary.head(10))


    ad_id                               car_name        price  \
0   17042   suzuki super carry truck 1.0 mt 2022  10080.99411   
1   73954        toyota innova g 2.0 at 2023 car  35830.03930   
2   74150         toyota corolla cross 1.8g 2023  30526.38376   
3   87573        toyota innova g 2.0 at 2022 car  34413.03209   
4   97011  suzuki super carry van blind van 2023  12105.29011   
5  182003            toyota innova e 2.0 mt 2022  30364.44008   
6  183963                 toyota vios e cvt 2022  21943.36870   
7  211394             toyota yaris g 1.5 at 2022  26315.84807   
8  304604             toyota vios g 1.5 cvt 2022  23967.66471   
9  485669            toyota innova e 2.0 mt 2022  30566.86968   

                   name       phone1  
0       Suzuki - Sydney  419552062.0  
1     Toyota - Brisbane  499151086.0  
2        Toyota - Perth  493783547.0  
3     Toyota - Adelaide  464535225.0  
4     Suzuki - Canberra      12345.0  
5      Toyota - Geelong  441675202.0  
6       Toy

# Data Visualisation